# Measles vaccination in the 1970 British Cohort Study

[Explanation of study here]

## 1. Install packages, load data, merge dataframes

In [ ]:
## Install packages
library(tidyverse)
library(ggplot2)
library(broom)
library(epitools)
library(gtsummary)
library(flextable)
library(patchwork)
library(gt)
library(svglite)
library(ggeffects)

In [ ]:
## Load data

## Link to data: https://unioxfordnexus-my.sharepoint.com/:f:/r/personal/wolf7427_ox_ac_uk/Documents/oxford_dphil/projects/measles/manuscript/input_tables?csf=1&web=1&e=UXloSR
base_path = "bsc70/input_tables"

files = c(
  "bcs1derived.tab", "bcs7072a.tab", "bcs7072b.tab", "bcs2derived.tab",
  "f699a.tab", "f699b.tab", "bcs3derived.tab", "sn3723.tab",
  "bcs4derived.tab", "bcs7016x.tab", "bcs_age46_main.tab",
  "bcs5derived.tab", "bcs96x.tab", "bcs6derived.tab", "bcs2000.tab",
  "bcs_2004_followup.tab", "bcs8derived.tab", "bcs_2008_followup.tab",
  "bcs70_2012_derived.tab", "bcs70_2012_flatfile.tab"
)

# Read and assign each file as a data frame with df_ prefix
for (file in files) {
    name = paste0("df_", tools::file_path_sans_ext(file))  # removes .tab extension
    path = file.path(base_path, file)
    assign(name, read_tsv(path))
}

In [ ]:
## Merge dataframes into appropriate sweep

##CHECK THE DATAFRAMES TO MAKE SURE YOU ACTUALLY NEED THEM. REMOVE UNNECCESSARY DATAFRAMES 

#Birth sweep
df_birth = df_bcs1derived %>% inner_join(df_bcs7072a, by = c("BCSID" = "bcsid"))
#nrow = 17194

#Age 5 sweep
df_age5 = df_bcs2derived %>% 
    inner_join(df_f699a, by = c("BCSID" = "bcsid")) %>%
    inner_join(df_f699b, by = c("BCSID" = "bcsid"))
#nrow = 13133

#Age 10 sweep
df_age10 = df_bcs3derived %>% inner_join(df_sn3723, by = "bcsid")
#nrow = 14868

#Age 16 sweep
df_age16 = df_bcs4derived %>% inner_join(df_bcs7016x, by = c("BCSID" = "bcsid"))
#nrow = 11614

#Age 26 sweep
df_age26 = df_bcs5derived %>% inner_join(df_bcs96x, by = c("BCSID" = "bcsid"))
#nrow = 9002

#Age 29 sweep
df_age29 = df_bcs6derived %>% inner_join(df_bcs2000, by = c("BCSID" = "bcsid"))
#nrow = 11076

#Age 34 sweep
df_age34 = df_bcs_2004_followup
#nrow = 9663

#Age 38 sweep
df_age38 = df_bcs8derived %>% inner_join(df_bcs_2008_followup, by = c("BCSID" = "bcsid"))
#nrow = 8873

#Age 42 sweep
df_age42 = df_bcs70_2012_derived %>% inner_join(df_bcs70_2012_flatfile, by = "BCSID")
#nrow = 9002

#Age 46 sweep
df_age46 = df_bcs_age46_main
#nrow = 9840


In [ ]:
##Merge all dataframes
df_merged = df_birth %>%
  full_join(df_age5, by = "BCSID") %>%
  full_join(df_age10, by = c("BCSID" = "bcsid")) %>%
  full_join(df_age16, by = "BCSID") %>%
  full_join(df_age26, by = "BCSID") %>%
  full_join(df_age29, by = "BCSID") %>%
  full_join(df_age34, by = c("BCSID" = "bcsid")) %>%
  full_join(df_age38, by = "BCSID") %>%
  full_join(df_age42, by = "BCSID") %>%
  full_join(df_age46, by = "BCSID")

In [ ]:
columns <- c(

  # IDENTIFIERS
  "BCSID",

  # SAMPLE INFORMATION
  "B10SAMPTAK",  # Blood sample obtained

  # BLOOD BIOMARKERS – Cardiometabolic
  "B10CHOL", "B10HDL", "B10HBA1C", "B10IGF1", "B10RTIN",
  "B10TRIG", "B10RBC",

  # BLOOD BIOMARKERS – Immune markers
  "B10HSCRP", "B10USCMG", "B10USCMM", "B10CMVAVC",

  # CARDIOMETABOLIC ENDPOINTS
  "b960455", "b960521", "diab", "dl1age", "kinddia1", "kinddia2", "kinddia3",
  "bd7hpb04", "b7diabag", "b8khpb03", "B9KHPB04", "B9PKHP04", "B10KHPB03",
  "B10DIABD",  # Diabetes

  "downhibp", "bp1age", "bd7hpb11", "b8khpb09", "B9KHPB09", "B9PKHP09",
  "B10KHPB10",  # Hypertension

  "B9PKHP19", "B10KHPB20",                              # Stroke
  "B10HRTP01",                                          # Heart attack
  "B9PKHP18", "B10HRTP02",                              # CHD
  "B10HRTP03",                                          # Angina
  "B10HRTP04",                                          # CHF
  "B10CLOTB",                                           # Clotting disorders
  "B10HEIGHTCM", "B10MWEIGHT",                          # BMI (age 46)

  # IMMUNE/INFECTION-RELATED ENDPOINTS
  "B9KHPB16", "B9PKHP16", "B10KHPB18",                  # Viral hepatitis
  "ob13.34", "cancer", "cancty01", "cancty02", "cancty03", "bd7hpb07",
  "b7ccty01", "b8khpb06", "b8ccty01", "B9KHPB07", "B9CCTY01", "B9PCCT01",
  "B10KHPB06", "B10CCTY01",                             # Leukemia

  "b8ccty02", "B9CCTY02", "B9PCCT02", "B10CCTY02",       # Hodgkin's lymphoma
  "b8ccty03", "B9CCTY03", "B9PCCT03", "B10CCTY03",       # Lymphoma

  "rd6c.1", "ob13.26", "ob13.27",                        # URTI / Chest infections
  "rc3.13", "meb4.61", "meb4.62", "meb4.63", "meb4.64", "meb4.65",  # Urinary
  "B10HPRB03", "B9PSKI05", "B9PHPR03", "B9HPRB03", "b8hprb03",      # Ear/skin

  "earprob1", "earprob2", "rc3.2", "meb4.6", "meb4.7", "meb4.8", 
  "meb4.9", "meb4.10",                                    # Ear infections

  "B10SKIN05", "B9SKIN05", "b8skin05",                   # Skin/fungal
  "cl1age10", "skincon1", "skincon2", "skincon3", "skincon4",
  "skincon5", "skincon6", "skincon7",

  "B10GYNP07", "B9GYNP07", "gynaep01", "gynaep02", "gynaep03", "gynaep04",
  "gynaep05", "gynaep06", "gynaep07", "gynaep08", "gynaep09", "gynaep10", # Pelvic/gynae

  "bladprb1", "bladprb2", "bladprb3", "bladprb4", "bladprb5",    # Nephritis
  "b8blkdx1", "B9BLKDX1", "B9PBLKD1", "B10BLKDX1",

  "B10BLKDD3", "B10BLKDX3", "B9PBLKD3", "B9BLKDX3", "b8blkdx3",  # Kidney/Bladder

  "e072", "b10.1",                                               # Other

  "meb4.36", "meb4.37", "meb4.38", "meb4.39", "meb4.40", 
  "ob10", "rc3.8", "b960449", "b960515", "hhfbane1", "hhfbane2", "hhfbane3",
  "hhfbane4", "hhfbane5", "cl1age3",                             # Bronchitis

  "meb4.31", "meb4.32", "meb4.33", "meb4.34", "meb4.35", "ob9.9", 
  "ob13.25", "rc3.7", "b8khpb01", "B9KHPB02", "B9PKHP02", "B10KHPB01", # Wheezy bronchitis

  "e073", "b11.13", "b11.14", "b11.15", "b11.16",
  "meb4.41", "meb4.42", "meb4.43", "meb4.44", "meb4.45", "rc3.9",      # Pneumonia

  "b12.3", "b12.4", "ob11.3",        # Mumps
  "b12.5", "b12.6", "ob11.4",        # Whooping cough
  "b12.7", "b12.8", "ob11.5",        # Chicken pox
  "e074", "b12.9", "b12.10", "ob11.6", # Meningitis

  # CHARACTERISTICS
  "a0255",         # Sex/gender
  "BD1REGN",       # Location/region
  "e245",          # Ethnic group
  "BD1PSOC",       # Social index

  # Previous pregnancies
  "a0055", "a0061", "a0067", "a0073", "a0079", "a0085", 
  "a0091", "a0102", "a0108", "a0114", "a0120", "a0126", 
  "a0132", "a0138", "a0149", "a0155", "a0161",
  "a0053", "a0059", "a0065", "a0071", "a0077", "a0083", 
  "a0089", "a0100", "a0106", "a0112", "a0118", "a0124", 
  "a0130", "a0136", "a0147", "a0153", "a0159",

  "BD1MAGE",       # Maternal age
  "a0043b",        # Smoking during pregnancy

  # EXPOSURES
  "e023a", "b14.7", "b15.5", "b15.6", "b15.7", "b15.8",   # Measles vaccination
  "b12.1", "b12.2", "ob11.2"                              # Measles infection
)


In [ ]:
df_selected = df_merged %>% select(all_of(columns))

## 2. Pre-processing

In [ ]:
df_process = df_selected

#### Exposure

In [ ]:
# ------------------------------------------------------------------------------
# MEASLES INFECTION AND AGE OF ONSET
#
# This section combines information from multiple sweeps to determine whether
# the child ever had measles and at what age.
#
# Variables used:
# - b12.1: Age 10 report — has child *ever* had measles?
#   - 1 = Yes → "Yes"
#   - 2 = No  → "No"
#   - Other / missing → NA
#
# - ob11.2: Age 16 report — has teen had measles *since age 10*?
#   - If this is 1 (Yes), overwrite prior response and set measlesinfection = "Yes"
#   - Also set measles_age = 16 to reflect timing
#
# Age at infection (measles_age):
# - b12.2: Reported age of measles infection at age 10
#   - 0 = Infancy → recoded to 0.5 years
#   - -3, -8, NA → set as NA
#
# Final output variables:
# - measlesinfection: Factor ("No", "Yes") indicating whether measles was ever reported
# - measles_age: Age (in years) of reported measles infection
# ------------------------------------------------------------------------------

## Infection
df_process$measlesinfection = ifelse(df_process$b12.1 == 1, "Yes",
                                ifelse(df_process$b12.1 == 2, "No", NA)) ## age 10 sweep: has child ever had measles

df_process$measlesinfection[df_process$ob11.2 == 1] = "Yes" ## age 16 sweep: teen has had measles since age 10

#Make into factor
df_process$measlesinfection = factor(df_process$measlesinfection, levels = c("No", "Yes"))

#Age at measles 10
df_process$measles_age = ifelse(
  df_process$b12.2 %in% c(-8, -3) | is.na(df_process$b12.2), NA,
  ifelse(df_process$b12.2 == 0, 0.5, df_process$b12.2)
)

df_process$measles_age[df_process$ob11.2 == 1] = 16

#Variables that are useful from this: measlesinfection and measles_age

#### Covariates

Sex, ethnicity, location, social index, maternal smoking, maternal age, and # of older children

In [ ]:
# ------------------------------------------------------------------------------
# COVARIATES DERIVATION
#
# This section derives covariates used in the analysis:
#
# 1. SEX:
#    - Derived from variable `a0255`
#    - Recoded as a factor with labels "Male" (0) and "Female" (1)
#
# 2. ETHNICITY:
#    - Derived from variable `e245`
#    - First recoded into 7 detailed categories
#    - Then collapsed into a binary variable: "European UK" (0) vs "Other" (1)
#
# 3. LOCATION:
#    - Derived from variable `BD1REGN`
#    - Recoded into 12 region names as a factor
#
# 4. SOCIAL INDEX:
#    - Derived from variable `BD1PSOC`
#    - Categorized into social classes (I to V and non-manual/manual distinctions)
#    - "Single/Other" group is removed and coded as NA
#    - Also converted to numeric (1–8) for ordinal analysis, with -1 (unknown) set to NA
#
# 5. MATERNAL SMOKING:
#    - Derived from variable `a0043b`
#    - Recoded as "Never" or "Ever" smoked
#
# 6. MATERNAL AGE:
#    - Directly taken from variable `BD1MAGE`
# ------------------------------------------------------------------------------


##COVARIATES

# SEX
df_process$sex <- factor(
  dplyr::case_when(
    df_process$a0255 == 1 ~ 0,     # Male → 0
    df_process$a0255 == 2 ~ 1,     # Female → 1
    df_process$a0255 %in% c(-1, -2, -3) ~ NA_real_  # Missing/unknown → NA
  ),
  levels = c(0, 1),
  labels = c("Male", "Female")
)


# ETHNICITY
df_process <- df_process %>%
  mutate(ethnicity = factor(case_when(
    e245 == 1  ~ "European UK",
    e245 == 2  ~ "European Other",
    e245 == 3  ~ "West Indian",
    e245 == 4  ~ "Indian-Pakistani",
    e245 == 5  ~ "Other Asian",
    e245 == 6  ~ "African",
    e245 == 7  ~ "Other",
    e245 %in% c(-1, -2, -3, -4) ~ NA_character_,
    TRUE ~ NA_character_
  ), levels = c(
    "European UK", "European Other", "West Indian", "Indian-Pakistani",
    "Other Asian", "African", "Other"
  )))

# ETHNICITY binary
df_process <- df_process %>%
  mutate(
    ethnicity = case_when(
      ethnicity == "European UK" ~ 0L,
      ethnicity %in% c("European Other", "West Indian", "Indian-Pakistani", "Other Asian", "African", "Other") ~ 1L,
      ethnicity == "Unknown" ~ NA_integer_
    ),
    ethnicity_binary = factor(ethnicity, levels = c(0, 1), labels = c("European UK", "Other"))
  )

# LOCATION
df_process <- df_process %>%
  mutate(location = factor(case_when(
    BD1REGN == 1  ~ "North",
    BD1REGN == 2  ~ "Yorks and Humberside",
    BD1REGN == 3  ~ "East Midlands",
    BD1REGN == 4  ~ "East Anglia",
    BD1REGN == 5  ~ "South East",
    BD1REGN == 6  ~ "South West",
    BD1REGN == 7  ~ "West Midlands",
    BD1REGN == 8  ~ "North West",
    BD1REGN == 9  ~ "Wales",
    BD1REGN == 10 ~ "Scotland",
    BD1REGN == 11 ~ "Northern Ireland",
    BD1REGN == 12 ~ "Overseas",
    BD1REGN %in% c(-1, -2) ~ NA_character_,
    TRUE ~ NA_character_
  ), levels = c(
    "North", "Yorks and Humberside", "East Midlands", "East Anglia",
    "South East", "South West", "West Midlands", "North West",
    "Wales", "Scotland", "Northern Ireland", "Overseas"
  ))
)

#SOCIAL INDEX
df_process <- df_process %>%
  mutate(
    socialindex = case_when(
      BD1PSOC %in% c(1, 2) ~ "Single/Other",
      BD1PSOC == 3 ~ "V unskilled",
      BD1PSOC == 4 ~ "IV partly-skilled",
      BD1PSOC == 5 ~ "III manual",
      BD1PSOC == 6 ~ "III non-manual",
      BD1PSOC == 7 ~ "II managerial/technical",
      BD1PSOC == 8 ~ "I professional",
      TRUE ~ NA_character_
    ),
    socialindex = factor(
      socialindex,
      levels = c("Single/Other", "V unskilled", "IV partly-skilled", "III manual", 
                 "III non-manual", "II managerial/technical", "I professional"),
      ordered = TRUE
    )
  )

df_process <- df_process %>%
  mutate(
    socialindex = ifelse(socialindex == "Single/Other", NA, as.character(socialindex)),
    socialindex = factor(socialindex, 
                         levels = c("V unskilled", "IV partly-skilled", "III manual", 
                                    "III non-manual", "II managerial/technical", "I professional"),
                         ordered = TRUE)
  )


#numeric coding from 1 (lowest) to 8 (highest):
df_process$socialindex_num = as.numeric(df_process$socialindex)

#mark unknowns as NA
df_process$socialindex[df_process$BD1PSOC == -1] <- NA
df_process$socialindex_num[df_process$BD1PSOC == -1] <- NA

#MATERNAL SMOKING
df_process = df_process %>%
  mutate(
    maternal_smoking = case_when(
      a0043b == 1 ~ "Never",
      a0043b %in% c(2, 3, 4, 5, 6) ~ "Ever",
      a0043b == -3 ~ NA_character_
    ),
    maternal_smoking = factor(maternal_smoking, levels = c("Never", "Ever"))
  )


#MATERNAL AGE
df_process$maternal_age = df_process$BD1MAGE

In [ ]:
# ------------------------------------------------------------------------------
# OLDER SIBLINGS DERIVATION
#
# This section derives two variables capturing the number of older siblings:
#
# - Source variables:
#     - `outcome_cols`: indicators for pregnancy outcomes (`a0055`, `a0061`, ..., `a0161`)
#                       where 1 indicates a live birth
#     - `sex_cols`: reported sex of each prior pregnancy (`a0053`, `a0059`, ..., `a0159`)
#                   valid if coded as 1–4 (i.e., not unknown or missing)
#
# - Logic:
#     - An older sibling is counted if:
#         - The pregnancy outcome was a live birth (value == 1)
#         - The sex variable is valid (1–4)
#
# - Variables created:
#     1. `older_siblings_raw`: numeric count of older siblings
#     2. `older_siblings_cat`: categorized version of the count with levels:
#         "0", "1", "2", "3", "4+"
#       This is stored as an ordered factor.
# ------------------------------------------------------------------------------


#OLDER SIBLINGS
outcome_cols <- c("a0055", "a0061", "a0067", "a0073", "a0079", "a0085", 
                  "a0091", "a0102", "a0108", "a0114", "a0120", "a0126", 
                  "a0132", "a0138", "a0149", "a0155", "a0161")

sex_cols <- c("a0053", "a0059", "a0065", "a0071", "a0077", "a0083", 
              "a0089", "a0100", "a0106", "a0112", "a0118", "a0124", 
              "a0130", "a0136", "a0147", "a0153", "a0159")

# Function to check whether a pregnancy counts as a sibling
count_siblings <- function(outcomes, sexes) {
  valid <- outcomes == 1 & sexes %in% 1:4
  sum(valid, na.rm = TRUE)
}

df_process <- df_process %>%
  rowwise() %>%
  mutate(
    older_siblings_raw = sum(
      c_across(all_of(outcome_cols)) == 1 &
      c_across(all_of(sex_cols)) %in% 1:4,
      na.rm = TRUE
    ),
    older_siblings_cat = case_when(
      older_siblings_raw == 0 ~ "0",
      older_siblings_raw == 1 ~ "1",
      older_siblings_raw == 2 ~ "2",
      older_siblings_raw == 3 ~ "3",
      older_siblings_raw >= 4 ~ "4+"
    )
  ) %>%
  ungroup() %>%
  mutate(
    older_siblings_cat = factor(older_siblings_cat, levels = c("0", "1", "2", "3", "4+"), ordered = TRUE)
  )

#### Outcomes

##### Blood biomarkers

In [ ]:
# ------------------------------------------------------------------------------
# BIOMARKERS CLEANING AND DERIVATION
#
# This section standardizes and interprets biomarker data collected at age 46.
#
# ▸ GENERAL CLEANING:
# - Applies to all blood biomarker variables:
#     "Refused" (-9), "Unsuitable" (-8), and "Not applicable" (-1) → set to NA
# - For COVID test result (CS_COV_RESULT), -8 and -1 also set to NA
#
# ▸ BIOMARKERS CLEANED:
#   "B10CHOL", "B10HDL", "B10HBA1C", "B10IGF1", "B10RTIN", "B10TRIG",
#   "B10RBC", "B10HSCRP", "B10USCMG", "B10USCMM", "B10CMVAVC", "CS_COV_RESULT"
#
# ▸ CMV IgG (B10USCMG):
# - Interpreted as binary:
#     - 2 = Positive → 1
#     - 1 or 3 = Negative or Unequivocal → 0
#     - Others → NA
# - Recoded as factor: "Negative", "Positive"
#
# ▸ BLOOD SAMPLE OBTAINED (B10SAMPTAK):
# - 1 = "Yes", 2 = "No"
# - -1, -8, -9 = missing → NA
# - Recoded as factor: "No", "Yes"
#
# This processing ensures biomarkers are usable in models and summary tables,
# with consistent NA handling and meaningful factor levels.
# ------------------------------------------------------------------------------

#Clean up biomarkers

#Make anything that was "Refused" (-9), "Unsuitable" (-8), or "Not applicable" (-1) into NA. 
#For COVID, "Void" (-8) and "Not applicable" (-1) are dropped.
bloodbiomarkers = c(
  "B10CHOL", "B10HDL", "B10HBA1C", "B10IGF1", "B10RTIN", "B10TRIG", "B10RBC",
  "B10HSCRP", "B10USCMG", "B10USCMM", "B10CMVAVC"
)

# Replace -9, -8, -1 with NA across these variables
df_process[bloodbiomarkers] = lapply(df_process[bloodbiomarkers], function(x) {
  x[x %in% c(-9, -8, -1)] = NA
  return(x)
})

##Categorical variables
#CMV IgG
df_process$B10USCMG = ifelse(df_process$B10USCMG == 2, 1,
                              ifelse(df_process$B10USCMG == 1, 0,
                              ifelse(df_process$B10USCMG == 3, 0, NA)))##consider unequivocal to be negative

df_process$B10USCMG = factor(df_process$B10USCMG,
                                     levels = c(0, 1),
                                     labels = c("Negative", "Positive"))

#blood sample obtained
df_process$bloodobtained <- case_when(
  df_process$B10SAMPTAK == 1 ~ "Yes",
  df_process$B10SAMPTAK == 2 ~ "No",
  df_process$B10SAMPTAK %in% c(-1, -8, -9) ~ NA_character_
)
df_process$bloodobtained <- factor(df_process$bloodobtained, levels = c("No", "Yes"))

In [ ]:
##blood
df_process = df_process %>%
  rename(
    chol     = B10CHOL,
    hdl      = B10HDL,
    hba1c    = B10HBA1C,
    igf1     = B10IGF1,
    frtin    = B10RTIN,
    trig     = B10TRIG,
    rbc      = B10RBC,
    crp      = B10HSCRP,
    cmvigg   = B10USCMG,
    cmvigm   = B10USCMM,
    cmvav    = B10CMVAVC
  )

##### Cardio and immunological/immune outcomes

In [ ]:
# ------------------------------------------------------------------------------
# assign_condition(): Assigns EVER (1), NEVER (0), or NA for a health condition
# using age 46 (B10) as the primary source, and earlier sweeps only to support
# the detection of EVER cases.
#
# Input:
# - positive_matrix: columns indicating positive responses (TRUE/FALSE), with the
#   age 46 (B10)/latest indicator in the first column, followed by prior sweeps.
# - negative_matrix: same structure, where TRUE = explicit negative (e.g., 0/2)
#
# Logic:
# - Assign 1 (EVER) if:
#     ▸ The age 46 value (B10) is positive (TRUE), OR
#     ▸ Any earlier wave indicates a positive response
#
# - Assign 0 (NEVER) if:
#     ▸ The age 46 value (B10) is explicitly negative (TRUE in negative_matrix[,1])
#
# - Assign NA if:
#     ▸ Age 46 is missing/unknown, AND
#     ▸ No prior sweep indicates a positive response
#
# This ensures:
# - B10 determines "NEVER" classification exclusively
# - Prior waves help rescue missing B10 data to identify "EVER"
# - Ambiguous or missing responses default to NA
# ------------------------------------------------------------------------------



assign_condition <- function(df, name, positive_matrix, negative_matrix) {
  n <- nrow(df)
  result <- rep(NA, n)
  
  # Extract age 46 values — first column
  b10_pos <- positive_matrix[, 1]
  b10_neg <- negative_matrix[, 1]

  # Check for any positive prior (if more than 1 column)
  prior_pos <- if (ncol(positive_matrix) > 1) {
    rowSums(positive_matrix[, -1, drop = FALSE], na.rm = TRUE) > 0
  } else {
    rep(FALSE, n)
  }

  # Assign 1 (EVER): age 46 positive OR any prior positive
  result[b10_pos == TRUE | prior_pos] <- 1

  # Assign 0 (NEVER): only if age 46 explicitly says NO
  result[is.na(result) & b10_neg == TRUE] <- 0

  # Assign as factor
  df[[name]] <- factor(result, levels = c(0, 1), labels = c(0, 1))
  return(df)
}



# Diabetes
diabetes_pos <- with(df_process, cbind(
    B10KHPB03 == 1,#wave 10 age 46 must be first
  diab == 1,
  bd7hpb04 == 1,
  b8khpb03 == 1,
  B9KHPB04 == 1,
  B9PKHP04 == 1,
  !is.na(dl1age) & !(dl1age %in% c(98, 99))
))

diabetes_neg <- with(df_process, cbind(
      B10KHPB03 == 0,
  diab %in% c(0, 2),
  bd7hpb04 == 0,
  b8khpb03 == 0,
  B9KHPB04 == 0,
  B9PKHP04 == 0,
  is.na(dl1age) | dl1age %in% c(98, 99)
))

df_process <- assign_condition(df_process, "diabetes", diabetes_pos, diabetes_neg)

# Hypertension
htn_pos <- with(df_process, cbind(
      B10KHPB10 == 1,
  downhibp == 1,
  bd7hpb11 == 1,
  b8khpb09 == 1,
  B9KHPB09 == 1,
  B9PKHP09 == 1,
  !is.na(bp1age) & !(bp1age %in% c(98, 99))
))
htn_neg <- with(df_process, cbind(
      B10KHPB10 == 0,
  downhibp %in% c(0, 2),
  bd7hpb11 == 0,
  b8khpb09 == 0,
  B9KHPB09 == 0,
  B9PKHP09 == 0,
  is.na(bp1age) | bp1age %in% c(98, 99)
))
df_process <- assign_condition(df_process, "hypertension", htn_pos, htn_neg)

# Stroke
stroke_pos <- with(df_process, cbind(B10KHPB20 == 1, B9PKHP19 == 1))
stroke_neg <- with(df_process, cbind(B10KHPB20 == 0, B9PKHP19 == 0))
df_process <- assign_condition(df_process, "stroke", stroke_pos, stroke_neg)

# CAD
cad_pos <- with(df_process, cbind(B10HRTP02 == 1, B9PKHP18 == 1))
cad_neg <- with(df_process, cbind(B10HRTP02 == 0, B9PKHP18 == 0))
df_process <- assign_condition(df_process, "cad", cad_pos, cad_neg)

# Heart Attack
ha_pos <- with(df_process, cbind(B10HRTP01 == 1))
ha_neg <- with(df_process, cbind(B10HRTP01 == 0))
df_process <- assign_condition(df_process, "heart_attack", ha_pos, ha_neg)

# Angina
angina_pos <- with(df_process, cbind(B10HRTP03 == 1))
angina_neg <- with(df_process, cbind(B10HRTP03 == 0))
df_process <- assign_condition(df_process, "angina", angina_pos, angina_neg)

# CHF
chf_pos <- with(df_process, cbind(B10HRTP04 == 1))
chf_neg <- with(df_process, cbind(B10HRTP04 == 0))
df_process <- assign_condition(df_process, "chf", chf_pos, chf_neg)

# Blood Clot
clot_pos <- with(df_process, cbind(B10CLOTB == 1))
clot_neg <- with(df_process, cbind(B10CLOTB == 2))
df_process <- assign_condition(df_process, "blood_clot_disorder", clot_pos, clot_neg)

# Liver Disease
liver_pos <- with(df_process, cbind(B10KHPB18 == 1, B9KHPB16 == 1, B9PKHP16 == 1))
liver_neg <- with(df_process, cbind(B10KHPB18 == 0, B9KHPB16 == 0, B9PKHP16 == 0))
df_process <- assign_condition(df_process, "liver_disease", liver_pos, liver_neg)

# Ear Infection
ear_pos <- with(df_process, cbind(
      B10HPRB03 == 1,
  meb4.6 == 1,
  meb4.7 == 1,
  meb4.8 == 1,
  rc3.2 %in% c(1, 2, 3, 6),
  earprob1 == 1,
  earprob2 == 2,
  b8hprb03 == 1,
  B9HPRB03 == 1,
  B9PHPR03 == 1
))
ear_neg <- with(df_process, cbind(
      B10HPRB03 == 0,
  meb4.9 == 1,
  rc3.2 == 4,
  B9HPRB03 == 0,
  B9PHPR03 == 0
))
df_process <- assign_condition(df_process, "ear_infection", ear_pos, ear_neg)

# Fungal Skin Infection
fungus_pos <- with(df_process, cbind(
    B10SKIN05 == 1,
  skincon1 == 5, skincon2 == 5, skincon3 == 5, skincon4 == 5,
  skincon5 == 5, skincon6 == 5, skincon7 == 5,
  b8skin05 == 1, B9SKIN05 == 1
))
fungus_neg <- with(df_process, cbind(
    B10SKIN05 == 0,
  skincon1 %in% c(1:4, 6:8, 98, 99), skincon2 %in% c(1:4, 6:8, 98, 99),
  skincon3 %in% c(1:4, 6:8, 98, 99), skincon4 %in% c(1:4, 6:8, 98, 99),
  skincon5 %in% c(1:4, 6:8, 98, 99), skincon6 %in% c(1:4, 6:8, 98, 99),
  skincon7 %in% c(1:4, 6:8, 98, 99),
  b8skin05 == 0, B9SKIN05 == 0
))
df_process <- assign_condition(df_process, "fungal_skin", fungus_pos, fungus_neg)

# Pelvic Infection
pelvic_pos <- with(df_process, cbind(
    B10GYNP07 == 1,
  gynaep01 == 2, gynaep02 == 2, gynaep03 == 2, gynaep04 == 2,
  gynaep05 == 2, gynaep06 == 2, gynaep07 == 2, gynaep08 == 2,
  gynaep09 == 2, gynaep10 == 2, B9GYNP07 == 1
))
pelvic_neg <- with(df_process, cbind(B9GYNP07 == 0, B10GYNP07 == 2))
df_process <- assign_condition(df_process, "pelvic_infection", pelvic_pos, pelvic_neg)

# Nephritis
nephritis_pos <- with(df_process, cbind(
    B10BLKDX1 == 1,
  bladprb1 == 1, bladprb2 == 1, bladprb3 == 1, bladprb4 == 1, bladprb5 == 1,
  b8blkdx1 == 1, B9BLKDX1 == 1, B9PBLKD1 == 1
))
nephritis_neg <- with(df_process, cbind(
    B10BLKDX1 == 0,
  b8blkdx1 == 0, B9BLKDX1 == 0, B9PBLKD1 == 0
))
df_process <- assign_condition(df_process, "nephritis", nephritis_pos, nephritis_neg)

# Kidney Infection
kidney_pos <- with(df_process, cbind(
    B10BLKDX3 == 1,
  bladprb1 == 3, bladprb2 == 3, bladprb3 == 3, bladprb4 == 3, bladprb5 == 3,
  b8blkdx3 == 1, B9BLKDX3 == 1, B9PBLKD3 == 1
))
kidney_neg <- with(df_process, cbind(
    B10BLKDX3 == 0,
  b8blkdx3 == 0, B9BLKDX3 == 0, B9PBLKD3 == 0
))
df_process <- assign_condition(df_process, "kidney_infection", kidney_pos, kidney_neg)

# Leukaemia
leukemia_pos <- with(df_process, cbind(
    B10CCTY01 == 1,
  cancty01 == 1, cancty02 == 1, cancty03 == 1,
  b7ccty01 == 1, b8ccty01 == 1, B9CCTY01 == 1, B9PCCT01 == 1
))
leukemia_neg <- with(df_process, cbind(
  B10CCTY01 %in% c(0, 2, 97),
  cancty01 %in% c(0, 2, 97),
  cancty02 %in% c(0, 2, 97),
  cancty03 %in% c(0, 2, 97),
  b7ccty01 %in% c(0, 2, 97),
  b8ccty01 %in% c(0, 2, 97),
  B9CCTY01 %in% c(0, 2, 97),
  B9PCCT01 %in% c(0, 2, 97)
))
df_process <- assign_condition(df_process, "leukaemia", leukemia_pos, leukemia_neg)

# Hodgkins
hodgkins_pos <- with(df_process, cbind(
      B10CCTY02 == 1,
  b7ccty01 == 2,
  b8ccty02 == 1,
  B9CCTY02 == 1,
  B9PCCT02 == 1
))
hodgkins_neg <- with(df_process, cbind(
      B10CCTY02 == 0,
  !b7ccty01 %in% 2,
  b8ccty02 == 0,
  B9CCTY02 == 0,
  B9PCCT02 == 0
))
df_process <- assign_condition(df_process, "hodgkins", hodgkins_pos, hodgkins_neg)

# Lymphoma
lymphoma_pos <- with(df_process, cbind(
      B10CCTY03 == 1,
  b7ccty01 == 3,
  b8ccty03 == 1,
  B9CCTY03 == 1,
  B9PCCT03 == 1
))
lymphoma_neg <- with(df_process, cbind(
      B10CCTY03 == 0,
  !b7ccty01 %in% 3,
  b8ccty03 == 0,
  B9CCTY03 == 0,
  B9PCCT03 == 0
))
df_process <- assign_condition(df_process, "lymphoma", lymphoma_pos, lymphoma_neg)

#### Non-age-46 (use latest wave instead)

# Mumps
mumps_pos <- with(df_process, cbind(ob11.3 == 1, b12.3 == 1))
mumps_neg <- with(df_process, cbind(ob11.3 == 2, b12.3 == 2))
df_process <- assign_condition(df_process, "mumps", mumps_pos, mumps_neg)

# Whooping Cough
whoop_pos <- with(df_process, cbind(ob11.4 == 1, b12.5 == 1))
whoop_neg <- with(df_process, cbind(ob11.4 == 2, b12.5 == 2))
df_process <- assign_condition(df_process, "whooping_cough", whoop_pos, whoop_neg)

# Chicken Pox
chickenpox_pos <- with(df_process, cbind(ob11.5 == 1, b12.7 == 1))
chickenpox_neg <- with(df_process, cbind(ob11.5 == 2, b12.7 == 2))
df_process <- assign_condition(df_process, "chicken_pox", chickenpox_pos, chickenpox_neg)

# Meningitis
meningitis_pos <- with(df_process, cbind(ob11.6 == 1, e074 %in% c(1, 2, 3, 4), b12.9 == 1))
meningitis_neg <- with(df_process, cbind(ob11.6 == 2, e074 == 5, b12.9 == 2))
df_process <- assign_condition(df_process, "meningitis", meningitis_pos, meningitis_neg)

# Bronchitis
bronchitis_pos <- with(df_process, cbind(
  rc3.8 %in% c(1, 2, 3, 6),
  ob10 == 1,
  b10.1 == 1,
  e072 %in% c(1, 2, 3, 4)
))
bronchitis_neg <- with(df_process, cbind(
  rc3.8 == 4,
  ob10 == 2,
  b10.1 == 2,
  e072 == 5
))
df_process <- assign_condition(df_process, "bronchitis", bronchitis_pos, bronchitis_neg)

# Pneumonia
pneumonia_pos <- with(df_process, cbind(
  rc3.9 %in% c(1, 2, 3, 6),
  b11.13 == 1,
  b11.14 == 1,
  meb4.41 == 1,
  meb4.42 == 1,
  meb4.43 == 1,
  e073 %in% c(1, 2, 3, 4)
))
pneumonia_neg <- with(df_process, cbind(
  rc3.9 == 4,
  e073 == 5
))
df_process <- assign_condition(df_process, "pneumonia", pneumonia_pos, pneumonia_neg)

# UTI
uti_pos <- with(df_process, cbind(
    rc3.13 %in% c(1, 2, 3, 6),
  meb4.61 == 1,
  meb4.62 == 1,
  meb4.63 == 1
))
uti_neg <- with(df_process, cbind(
    rc3.13 == 4,
  meb4.64 == 1
))
df_process <- assign_condition(df_process, "uti", uti_pos, uti_neg)


In [ ]:
# ------------------------------------------------------------------------------
# GROUPED CONDITIONS AND DERIVED VARIABLES
#
# These blocks create composite health indicators by combining related conditions.
# Each grouped variable (e.g., heart_disease, blood_cancer) is coded as:
#
# - 1 (YES / EVER) if ANY of the component conditions are coded as 1 (i.e., EVER)
# - 0 (NO / NEVER) if ALL components are known and coded as 0 (i.e., NEVER)
# - NA (MISSING) if none are positive and at least one component is missing
#
# This logic preserves uncertainty: group-level NA is used only when it's unclear
# whether the condition ever occurred due to missing data in the components.
#
# Groupings include:
# - blood_cancer: leukaemia, hodgkins, lymphoma
# - heart_disease: cad, angina, chf, heart_attack
# - respiratory_infection: bronchitis or pneumonia
# - childhood_infections: mumps, whooping_cough, chicken_pox, meningitis
# - renal_disease: nephritis or kidney_infection
#
# BMI:
# - Body Mass Index is calculated as weight (kg) / height^2 (m^2)
# - Uses B10MWEIGHT and B10HEIGHTCM (from age 46)
# - Missing if either variable is unavailable or invalid
# ------------------------------------------------------------------------------


# Blood Cancer
df_process$blood_cancer <- factor(ifelse(
  rowSums(with(df_process, cbind(leukaemia == 1, hodgkins == 1, lymphoma == 1)), na.rm = TRUE) > 0, 1,
  ifelse(rowSums(with(df_process, cbind(leukaemia == 0, hodgkins == 0, lymphoma == 0)), na.rm = TRUE) == 3, 0, NA)
), levels = c(0, 1), labels = c(0, 1))


# Heart disease
df_process$heart_disease <- factor(ifelse(
  rowSums(with(df_process, cbind(cad == 1, angina == 1, chf == 1, heart_attack == 1)), na.rm = TRUE) > 0, 1,
  ifelse(rowSums(with(df_process, is.na(cbind(cad, angina, chf, heart_attack)))) == 0 &
         rowSums(with(df_process, cbind(cad == 0, angina == 0, chf == 0, heart_attack == 0))) == 4, 0, NA)),
  levels = c(0, 1), labels = c(0, 1))

# Respiratory Infections (bronchitis or pneumonia)
df_process$respiratory_infection <- factor(ifelse(
  rowSums(with(df_process, cbind(bronchitis == 1, pneumonia == 1)), na.rm = TRUE) > 0, 1,
  ifelse(rowSums(with(df_process, cbind(bronchitis == 0, pneumonia == 0)), na.rm = TRUE) == 2, 0, NA)),
  levels = c(0, 1), labels = c(0, 1))

# Childhood Infections (mumps, whooping cough, chickenpox, meningitis)
df_process$childhood_infections <- factor(ifelse(
  rowSums(with(df_process, cbind(mumps == 1, whooping_cough == 1, chicken_pox == 1, meningitis == 1)), na.rm = TRUE) > 0, 1,
  ifelse(rowSums(with(df_process, cbind(mumps == 0, whooping_cough == 0, chicken_pox == 0, meningitis == 0)), na.rm = TRUE) == 4, 0, NA)),
  levels = c(0, 1), labels = c(0, 1))

# Renal Disease (nephritis or kidney infection)
df_process$renal_disease <- factor(ifelse(
  rowSums(with(df_process, cbind(nephritis == 1, kidney_infection == 1)), na.rm = TRUE) > 0, 1,
  ifelse(rowSums(with(df_process, cbind(nephritis == 0, kidney_infection == 0)), na.rm = TRUE) == 2, 0, NA)),
  levels = c(0, 1), labels = c(0, 1))

# BMI
df_process$bmi <- with(df_process, ifelse(
  !is.na(B10HEIGHTCM) & B10HEIGHTCM > 0 &
  !is.na(B10MWEIGHT) & B10MWEIGHT > 0,
  
  B10MWEIGHT / ( (B10HEIGHTCM / 100)^2 ),  # BMI calculation
  
  NA  # If height or weight is missing/invalid
))

#### Clean

In [ ]:
head(df_process)

In [ ]:
##CLEAN AND SAVE DATAFRAME
clean_names = c(
    "BCSID", #id
    
    "chol", "hdl", "hba1c", "igf1", "frtin", "trig", "rbc", "crp", "cmvigg", "bloodobtained",#blood
    
    "measlesinfection", "measles_age", #exposure
    
    "sex", "ethnicity", "location", "socialindex", "socialindex_num", 
    "older_siblings_cat", "maternal_smoking", "maternal_age", "ethnicity_binary", #covariates
    
    "diabetes", "hypertension", "stroke", "cad", "heart_attack", 
    "angina", "chf", "blood_clot_disorder", "heart_disease", "bmi", #cardiometabolic outcomes
    
    "liver_disease", "leukaemia", "hodgkins", "lymphoma", "blood_cancer", 
    "bronchitis", "pneumonia", "uti", "ear_infection", "fungal_skin", 
    "pelvic_infection", "nephritis", "kidney_infection", "mumps", 
    "whooping_cough", "chicken_pox", "meningitis", 
    "respiratory_infection", "childhood_infections", "renal_disease" #immune/infection outcomes
)

In [ ]:
df_cleaned = df_process %>% select(all_of(clean_names))

In [ ]:
summary(df_cleaned)

## 3. Analyses

In [ ]:
covariates <- c("measlesinfection", "sex", "ethnicity_binary", "location", "socialindex", "older_siblings_cat",
                "maternal_age", "maternal_smoking")

In [ ]:
# Complete sample
df_full <- df_cleaned %>%
  filter(!is.na(measlesinfection), location != "Northern Ireland")

# Analytic sample (only complete covariate cases)
df_analytic <- df_full %>%
  filter(complete.cases(across(all_of(covariates))))

# Blood sample
bloodvars <- c("chol", "hdl", "hba1c", "igf1", "frtin", "trig", "rbc", "crp", "cmvigg")

df_blood <- df_analytic %>%
  filter(rowSums(!is.na(across(all_of(bloodvars)))) > 0)

#### Table 1: Characteristics

In [ ]:
### Table 1 for full sample
table1_full <- df_full %>%
  select(all_of(covariates), measlesinfection) %>%
  tbl_summary(
    by = measlesinfection,
    statistic = all_continuous() ~ "{mean} ({sd})",
    missing = "ifany"
  ) %>%
  add_overall() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles Status**") %>%
  bold_labels() %>%
  modify_caption("**Table 1A. Full Sample**")

### Save tables
table1_full %>%
  as_gt() %>%
  gtsave("table1_full.html")
table1_full_df <- as.data.frame(table1_full)
write.csv(table1_full_df, "table1_full.csv", row.names = FALSE)

In [ ]:
### Table 1 for analytic sample
table1_analytic <- df_analytic %>%
  select(all_of(covariates), measlesinfection) %>%
  tbl_summary(
    by = measlesinfection,
    statistic = all_continuous() ~ "{mean} ({sd})",
    missing = "no"
  ) %>%
  add_overall() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles Status**") %>%
  bold_labels() %>%
  modify_caption("**Table 1B. Analytic Sample**")

### Save tables
table1_analytic %>%
  as_gt() %>%
  gtsave("table1_analytic.html")
table1_analytic_df <- as.data.frame(table1_analytic)
write.csv(table1_analytic_df, "table1_analytic.csv", row.names = FALSE)

In [ ]:
# Table 1 for samples 
table1_blood <- df_blood %>%
  select(all_of(covariates), measlesinfection) %>%
  tbl_summary(
    by = measlesinfection,
    statistic = all_continuous() ~ "{mean} ({sd})",
    missing = "ifany"
  ) %>%
  add_overall() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles Status**") %>%
  bold_labels() %>%
  modify_caption("**Table 1C. Table 1 for those who provided blood sample by Measles Infection Status**")

### Save tables
table1_blood %>%
  as_gt() %>%
  gtsave("table1_blood.html")
table1_blood_df <- as.data.frame(table1_blood)
write.csv(table1_blood_df, "table1_blood.csv", row.names = FALSE)

#### Table 2: Outcomes by measles infection status

Complete cohort

In [ ]:
table_vars <- c(
  "chol", "hdl", "hba1c", "igf1", "frtin", "trig", "rbc", "crp", "cmvigg",  # continuous
  "diabetes", "hypertension", "stroke", "cad", "heart_attack", 
  "angina", "chf", "blood_clot_disorder", "heart_disease", "bmi",         # mix
  "liver_disease", "leukaemia", "hodgkins", "lymphoma", "blood_cancer", 
  "bronchitis", "pneumonia", "uti", "ear_infection", "fungal_skin", 
  "pelvic_infection", "nephritis", "kidney_infection", "mumps", 
  "whooping_cough", "chicken_pox", "meningitis", 
  "respiratory_infection", "childhood_infections", "renal_disease"
)


In [ ]:
# Table 2 for analytic set
table2_analytic <- df_analytic %>%
  select(all_of(c("measlesinfection", table_vars))) %>%
  tbl_summary(
    by = measlesinfection,
    statistic = all_continuous() ~ "{mean} ({sd})",  # show mean (SD)
    missing = "ifany"
  ) %>%
  add_overall() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles status**") %>%
  bold_labels()

### Save tables
table2_analytic %>%
  as_gt() %>%
  gtsave("table2.html")
table2_analytic_df <- as.data.frame(table2_analytic)
write.csv(table2_analytic_df, "table2_analytic.csv", row.names = FALSE)

In [ ]:
# Table 2 for blood set
table2_blood <- df_blood %>%
  select(all_of(c("measlesinfection", table_vars))) %>%
  tbl_summary(
    by = measlesinfection,
    statistic = all_continuous() ~ "{mean} ({sd})",  # show mean (SD)
    missing = "ifany"
  ) %>%
  add_overall() %>%
  modify_header(label = "**Characteristic**") %>%
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Measles status**") %>%
  bold_labels()

### Save tables
table2_blood %>%
  as_gt() %>%
  gtsave("table2_blood.html")
table2_blood_df <- as.data.frame(table2_blood)
write.csv(table2_blood_df, "table2_blood.csv", row.names = FALSE)

#### Figure 1: Association between infection and outcomes

##### Blood biomarkers transformations

In [ ]:
checkblood = c("chol", "hdl", "hba1c", "igf1", "frtin", "trig", "rbc", "crp")

In [ ]:
label_map <- c(

  # ID
  "BCSID" = "BCSID",

  # Blood markers (raw and standardized/log-transformed)
  "bloodobtained"    = "Blood Obtained",
  "chol"             = "Cholesterol",
  "hdl"              = "HDL",
  "hba1c"            = "HbA1c",
  "igf1"             = "IGF-1",
  "frtin"            = "Ferritin",
  "trig"             = "Triglycerides",
  "rbc"              = "Red Blood Cell Count",
  "crp"              = "C-Reactive Protein",
  "cmvigg"           = "CMV IgG",

  "log_frtin"        = "log(Ferritin)",
  "log_trig"         = "log(Triglycerides)",
  "log_crp"          = "log(CRP)",

  "chol_std"         = "Cholesterol",
  "hdl_std"          = "HDL",
  "hba1c_std"        = "HbA1c",
  "igf1_std"         = "IGF-1",
  "log_frtin_std"    = "log(Ferritin)",
  "log_trig_std"     = "log(Triglycerides)",
  "rbc_std"          = "Red Blood Cell Count",
  "log_crp_std"      = "log(CRP)",

  # Exposure
  "measlesinfection" = "Measles Infection",
  "measles_age"      = "Age at Measles",

  # Covariates
  "sex"              = "Sex",
  "ethnicity"        = "Ethnicity",
  "ethnicity_binary" = "Ethnicity",
  "location"         = "Location",
  "socialindex"      = "Social Index (Categorical)",
  "socialindex_num"  = "Social Index (Numeric)",
  "older_siblings_cat" = "Older Children",
  "maternal_smoking" = "Maternal Smoking",
  "maternal_age"     = "Maternal Age",

  # Cardiometabolic outcomes
  "diabetes"         = "Diabetes",
  "hypertension"     = "Hypertension",
  "stroke"           = "Stroke",
  "cad"              = "Coronary Artery Disease (CAD)",
  "heart_attack"     = "Heart Attack",
  "angina"           = "Angina",
  "chf"              = "Congestive Heart Failure",
  "blood_clot_disorder" = "Blood Clotting Disorder",
  "heart_disease"    = "Heart Disease",
  "bmi"              = "Body Mass Index (BMI)",

  # Immune / Infection outcomes
  "liver_disease"       = "Liver Disease",
  "leukaemia"           = "Leukaemia",
  "hodgkins"            = "Hodgkin's Lymphoma",
  "lymphoma"            = "Lymphoma",
  "blood_cancer"        = "Blood Cancer",
  "bronchitis"          = "Bronchitis",
  "pneumonia"           = "Pneumonia",
  "uti"                 = "Urinary Tract Infection (UTI)",
  "ear_infection"       = "Ear Infection",
  "fungal_skin"         = "Fungal Skin Infection",
  "pelvic_infection"    = "Pelvic Infection",
  "nephritis"           = "Nephritis",
  "kidney_infection"    = "Kidney Infection",
  "mumps"               = "Mumps",
  "whooping_cough"      = "Whooping Cough",
  "chicken_pox"         = "Chickenpox",
  "meningitis"          = "Meningitis",
  "respiratory_infection" = "Respiratory Infection",
  "childhood_infections"  = "Childhood Infections",
  "renal_disease"         = "Renal Disease"
)

In [ ]:
# Log transform CRP, FRTIN, and TRIG
df_blood <- df_blood %>%
  mutate(
    log_crp = log1p(crp),
    log_frtin = log1p(frtin),
    log_trig = log1p(trig)
  )

continuousblood = c("chol", "hdl", "hba1c", "igf1", "log_frtin", "log_trig", "rbc", "log_crp")

In [ ]:
# Create standardized versions with _std suffix
df_blood[paste0(continuousblood, "_std")] <- lapply(df_blood[continuousblood], scale)

continuousblood_std = c("chol_std", "hdl_std", "hba1c_std", "igf1_std", "log_frtin_std", "log_trig_std", "rbc_std", "log_crp_std")

##### Regressions

In [ ]:
covariates <- c("measlesinfection", "sex", "ethnicity_binary", "location", "socialindex", "older_siblings_cat",
                "maternal_age", "maternal_smoking")

In [ ]:
#### ggplot theme
my_custom_theme <- function(base_size = 12, base_family = "Helvetica") {
  theme_minimal(base_size = base_size, base_family = base_family) +
    theme(
      # Text
      plot.title = element_text(size = base_size + 4, face = "bold", hjust = 0.5),
      axis.title = element_text(size = base_size + 2),
      axis.text = element_text(size = base_size, color = "black"),
      axis.ticks = element_line(color = "black"),  # Optional: show ticks

      # Grid and Background
      panel.grid = element_blank(),
      panel.background = element_blank(),
      plot.background = element_blank(),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.8),

      # Legend
      legend.position = "bottom",
      legend.title = element_text(face = "bold"),
      legend.text = element_text(size = base_size),  # Optional

      # Facet strips
      strip.text = element_text(face = "bold", size = base_size),

      # Margin
     # plot.margin = margin(10, 10, 10, 10)  # Optional
    )
}


In [ ]:
regression_results <- lapply(continuousblood_std, function(var) {
  
  # --- Unadjusted model ---
  formula_unadjusted <- as.formula(paste(var, "~ measlesinfection"))
  model_unadjusted <- lm(formula_unadjusted, data = df_blood)
  n_unadjusted <- glance(model_unadjusted)$nobs
  
  unadjusted_result <- tidy(model_unadjusted, conf.int = TRUE) %>%
    mutate(
      model = "unadjusted",
      outcome = var,
      n = n_unadjusted,
      significant = if_else(
        grepl("measlesinfectionYes", term) & (conf.low > 0 | conf.high < 0),
        TRUE, FALSE
      ),
      outcome_label = label_map[outcome]
    )
  
  # --- Adjusted model ---
  formula_adjusted <- as.formula(paste(var, "~", paste(covariates, collapse = " + ")))
  model_adjusted <- lm(formula_adjusted, data = df_blood)
  n_adjusted <- glance(model_adjusted)$nobs
  
  adjusted_result <- tidy(model_adjusted, conf.int = TRUE) %>%
    mutate(
      model = "adjusted",
      outcome = var,
      n = n_adjusted,
      significant = if_else(
        grepl("measlesinfectionYes", term) & (conf.low > 0 | conf.high < 0),
        TRUE, FALSE
      ),
      outcome_label = label_map[outcome]
    )
  
  # Combine both
  bind_rows(unadjusted_result, adjusted_result)
})

# Combine all model results into one dataframe
regression_table <- bind_rows(regression_results)


In [ ]:
regression_table %>%
  filter(term == "measlesinfectionYes")

write.csv(regression_table, "blood_table.csv", row.names = FALSE)

In [ ]:
bloodcontinuousplot = regression_table %>%
  filter(term == "measlesinfectionYes", model == "adjusted") %>%
  ggplot(aes(x = estimate, y = reorder(outcome_label, estimate), color = significant)) +
  geom_errorbarh(
    aes(xmin = estimate - std.error * 1.96,
        xmax = estimate + std.error * 1.96),
    height = 0,
    linewidth = 0.7
  ) +
  geom_point(size = 3) +  # Remove shape/fill/color overrides
  geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
  scale_color_manual(values = c(`TRUE` = "darkslategray3", `FALSE` = "gray80")) +
  labs(
    title = "Effect of Measles on Continuous Blood Outcomes",
    x = "Beta Coefficient (95% CI)",
    y = "Outcome",
    color = "Significant"  # legend title
  ) +
  
  my_custom_theme()

ggsave("bloodcontinuousplot.svg", plot = bloodcontinuousplot, device = svglite, width = 6, height = 4)
ggsave("bloodcontinuousplot.pdf", plot = bloodcontinuousplot, width = 7, height = 4)

In [ ]:
bloodcontinuousplot

Categorical blood biomarkers

In [ ]:
# --- Unadjusted model ---
model_cmv_unadjusted <- glm(cmvigg ~ measlesinfection, data = df_blood, family = binomial)

cmvigg_unadj_results <- tidy(model_cmv_unadjusted, exponentiate = TRUE, conf.int = TRUE) %>%
  mutate(
    model = "unadjusted",
    n = nobs(model_cmv_unadjusted),
    significant = term == "measlesinfectionYes" & (conf.low > 1 | conf.high < 1)
  )

# --- Adjusted model ---
model_cmv_adjusted <- glm(
  cmvigg ~ measlesinfection + sex + ethnicity_binary + location + socialindex +
    older_siblings_cat + maternal_age + maternal_smoking,
  data = df_blood,
  family = binomial
)

cmvigg_adj_results <- tidy(model_cmv_adjusted, exponentiate = TRUE, conf.int = TRUE) %>%
  mutate(
    model = "adjusted",
    n = nobs(model_cmv_adjusted),
    significant = term == "measlesinfectionYes" & (conf.low > 1 | conf.high < 1)
  )

# Combine results
cmvigg_results <- bind_rows(cmvigg_unadj_results, cmvigg_adj_results)

In [ ]:
cmvigg_results %>%
  filter(term == "measlesinfectionYes")

write.csv(cmvigg_results, "cmvigg_table.csv", row.names = FALSE)

In [ ]:
bloodcategoricalplot = cmvigg_results %>%
  filter(term == "measlesinfectionYes", model == "adjusted") %>%
  ggplot(aes(y = term, x = estimate, xmin = conf.low, xmax = conf.high, color = significant)) +

  # Horizontal error bars
  geom_errorbarh(height = 0, linewidth = 0.8) +

  # Point estimate
  geom_point(size = 4) +

  # Reference line at OR = 1
  geom_vline(xintercept = 1, linetype = "dashed", color = "black") +

  # Custom color scale
  scale_color_manual(values = c(`TRUE` = "darkorange1", `FALSE` = "gray80")) +

  # Labels
  labs(
    title = "Odds Ratios for CMV IgG Positivity",
    x = "Odds Ratio (95% CI)",
    y = "Predictor",
    color = "Significant"
  ) +

  my_custom_theme()

ggsave("bloodcategoricalplot.svg", plot = bloodcategoricalplot, device = svglite, width = 6, height = 4)
ggsave("bloodcategoricalplot.pdf", plot = bloodcategoricalplot, width = 7, height = 2)

In [ ]:
bloodcategoricalplot

In [ ]:
cmvigg_pred <- ggpredict(model_cmv_adjusted, terms = "measlesinfection") %>% as.data.frame()

In [ ]:
cmvigg_table <- cmvigg_pred %>%
  select(
    `Measles Infection` = x,
    `Predicted Probability` = predicted,
    `95% CI Lower` = conf.low,
    `95% CI Upper` = conf.high
  )

write.csv(cmvigg_table, "cmvigg_pred_table.csv", row.names = FALSE)

In [ ]:
cmvigg_pred <- cmvigg_pred %>%
  mutate(measles_status = factor(x))  # Rename to avoid confusion

cmvigg_pred_plot = ggplot(cmvigg_pred, aes(y = measles_status, x = predicted, color = measles_status)) +
  geom_point(size = 3) +
  geom_errorbarh(aes(xmin = conf.low, xmax = conf.high), height = 0.0) +
  scale_color_manual(values = c("No" = "darkslategray3", "Yes" = "darkorange1")) +
  labs(
    title = "Predicted Probability of CMV IgG Positivity",
    subtitle = "Marginal effects by Measles Infection Status",
    y = "Measles Infection",
    x = "Predicted Probability",
    color = "Measles Infection"
  ) +
  my_custom_theme()

ggsave("cmvigg_pred_plot.svg", plot = cmvigg_pred_plot, device = svglite, width = 6, height = 4)
ggsave("cmvigg_pred_plot.pdf", plot = cmvigg_pred_plot, width = 7, height = 3)

In [ ]:
cmvigg_pred_plot

Endpoints

Categorical endpoints

In [ ]:
covariates <- c("measlesinfection", "sex", "ethnicity_binary", "location", "socialindex", "older_siblings_cat",
                "maternal_age", "maternal_smoking")

In [ ]:
categorical_outcomes <- c(
  "diabetes", "hypertension", "stroke", "cad", "heart_attack", 
  "angina", "chf", "blood_clot_disorder", "heart_disease",
  "liver_disease","leukaemia", "hodgkins", "lymphoma", "blood_cancer",
  "bronchitis", "pneumonia", "uti", "ear_infection",
  "pelvic_infection", "nephritis", "kidney_infection", "mumps", 
  "whooping_cough", "chicken_pox", "meningitis", 
  "respiratory_infection", "childhood_infections", "renal_disease"
)
#

In [ ]:
logistic_results <- list()
predicted_probs <- list()

for (var in categorical_outcomes) {

  # Conditionally modify covariates
  current_covariates <- if (var == "pelvic_infection") {
    setdiff(covariates, "sex")
  } else {
    covariates
  }

  # Define variable list
  predictors <- c("measlesinfection", current_covariates)

  # Build adjusted and unadjusted formulas
  form_adjusted <- as.formula(paste(var, "~", paste(predictors, collapse = " + ")))
  form_unadjusted <- as.formula(paste(var, "~ measlesinfection"))

  # Prepare modeling dataset
  df_model <- df_analytic %>%
    select(all_of(c(var, "measlesinfection", current_covariates))) %>%
    na.omit() %>%
    droplevels()

  # Skip if outcome has only one class
  if (nlevels(factor(df_model[[var]])) < 2) {
    warning(paste("Skipping", var, "- outcome has only one level"))
    next
  }

  # Skip if any covariate has only one level
  one_level_factors <- sapply(df_model[c("measlesinfection", current_covariates)], function(x) {
    is.factor(x) && nlevels(x) < 2
  })
  if (any(one_level_factors)) {
    warning(paste("Skipping", var, "- single-level factor(s):",
                  paste(names(which(one_level_factors)), collapse = ", ")))
    next
  }

  # --- Unadjusted model ---
  model_unadj <- glm(form_unadjusted, data = df_model, family = binomial)
  tidy_unadj <- broom::tidy(model_unadj, exponentiate = TRUE, conf.int = TRUE) %>%
    mutate(
      model = "unadjusted",
      outcome = var,
      n = nobs(model_unadj),
      significant = term == "measlesinfectionYes" & (conf.low > 1 | conf.high < 1)
    )

  # --- Adjusted model ---
  model_adj <- glm(form_adjusted, data = df_model, family = binomial)
  tidy_adj <- broom::tidy(model_adj, exponentiate = TRUE, conf.int = TRUE) %>%
    mutate(
      model = "adjusted",
      outcome = var,
      n = nobs(model_adj),
      significant = term == "measlesinfectionYes" & (conf.low > 1 | conf.high < 1)
    )

  # Predicted probabilities for adjusted model
  pred_df <- ggpredict(model_adj, terms = "measlesinfection") %>%
    as.data.frame() %>%
    mutate(outcome = var)

  # Combine and save
  logistic_results[[var]] <- bind_rows(tidy_unadj, tidy_adj)
  predicted_probs[[var]] <- pred_df
}


In [ ]:
# Combine all results into single tables
or_table <- bind_rows(logistic_results)
pred_table <- bind_rows(predicted_probs)

write.csv(or_table, "outcomes_or_table.csv", row.names = FALSE)
write.csv(pred_table, "outcomes_pred_table.csv", row.names = FALSE)

##### Predicted probability

In [ ]:
# Ensure label_map is correctly defined and matches outcome names
# label_map <- c("outcome1" = "Label 1", "outcome2" = "Label 2", ...)

plot_data <- pred_table %>%
  filter(x %in% c("Yes", "No"), !outcome %in% c("chf", "angina", "leukaemia", "cad", "lymphoma", "hodgkins", 
                                                "heart_attack", "childhood_infection", "renal_disease", "respiratory_infection")) %>%
  select(
    measles_status = x,
    predicted,
    conf.low,
    conf.high,
    outcome
  ) %>%
  mutate(
    outcome_label = label_map[outcome],
    measles_status = factor(measles_status, levels = c("No", "Yes"))  # Ensure correct order
  )

outcomescategoricalpred <- ggplot(plot_data, aes(x = predicted, y = reorder(outcome_label, predicted), color = measles_status)) +

  # 95% CI error bars
 # geom_errorbarh(aes(xmin = conf.low, xmax = conf.high), height = 0, linewidth = 0.8) +

  # Point estimates
  geom_point(size = 3) +

  # Vertical reference line at 0.0
  geom_vline(xintercept = 0.0, linetype = "dashed", color = "black") +

  # Labels and title
  labs(
    x = "Predicted Probability (95% CI)",
    y = "Outcome",
    color = "Measles Infection",
    title = "Predicted Probabilities by Measles Infection Status"
  ) +

  my_custom_theme()


In [ ]:
outcomescategoricalpred

In [ ]:
ggsave("outcomescategoricalpred.svg", plot = outcomescategoricalpred, device = svglite, width = 6, height = 4)
ggsave("outcomescategoricalpred.pdf", plot = outcomescategoricalpred, width = 7, height = 7)

##### Odds ratios

In [ ]:
or_table %>%
  filter(term == "measlesinfectionYes") %>%
  select(outcome, estimate, conf.low, conf.high, p.value, n, significant, term, model) 

In [ ]:
forest_df <- or_table %>%
select(outcome, estimate, conf.low, conf.high, p.value, significant, n, term, model) %>% mutate(outcome_label = label_map[outcome]) %>%
  filter(term == "measlesinfectionYes", !outcome %in% c("chf", "angina", "leukaemia", "cad", "lymphoma", "hodgkins", 
                                                "heart_attack", "childhood_infection", "renal_disease", "respiratory_infection"), 
                                             model == "adjusted")
  

In [ ]:
outcomescategoricalor = ggplot(forest_df, aes(x = estimate, y = reorder(outcome_label, estimate), color = significant)) +
  geom_errorbarh(aes(xmin = conf.low, xmax = conf.high),
                 height = 0,
                 linewidth = 0.8) +
  geom_point(size = 3) +
  geom_vline(xintercept = 1, linetype = "dashed", color = "black") +
  scale_color_manual(values = c(`TRUE` = "darkslategray3", `FALSE` = "gray80")) +
  labs(
    title = "Odds Ratios for Measles Infection Across Outcomes",
    x = "Odds Ratio (95% CI)",
    y = "Outcome",
    color = "Significant"
  ) +
  my_custom_theme()


In [ ]:
outcomescategoricalor

In [ ]:
ggsave("outcomescategoricalor.svg", plot = outcomescategoricalor, device = svglite, width = 6, height = 4)
ggsave("outcomescategoricalor.pdf", plot = outcomescategoricalor, width = 7, height = 7)

In [ ]:
#Continuous endpoints (BMI)

In [ ]:
covariates <- c("measlesinfection", "sex", "ethnicity_binary", "location", "socialindex", "older_siblings_cat",
                "maternal_age", "maternal_smoking")

In [ ]:
# Define the outcome
outcome <- "bmi"

# --- Unadjusted model ---
formula_unadjusted <- as.formula(paste(outcome, "~ measlesinfection"))
model_unadjusted <- lm(formula_unadjusted, data = df_analytic)
n_unadjusted <- glance(model_unadjusted)$nobs

bmi_unadj_result <- tidy(model_unadjusted, conf.int = TRUE) %>%
  mutate(
    model = "unadjusted",
    outcome = outcome,
    n = n_unadjusted,
    significant = term == "measlesinfectionYes" & (conf.low > 0 | conf.high < 0),
    outcome_label = label_map[outcome]
  )

# --- Adjusted model ---
formula_adjusted <- as.formula(paste(outcome, "~", paste(covariates, collapse = " + ")))
model_adjusted <- lm(formula_adjusted, data = df_analytic)
n_adjusted <- glance(model_adjusted)$nobs

bmi_adj_result <- tidy(model_adjusted, conf.int = TRUE) %>%
  mutate(
    model = "adjusted",
    outcome = outcome,
    n = n_adjusted,
    significant = term == "measlesinfectionYes" & (conf.low > 0 | conf.high < 0),
    outcome_label = label_map[outcome]
  )

# Combine both results
bmi_regression_result <- bind_rows(bmi_unadj_result, bmi_adj_result)
write.csv(bmi_regression_result, "bmi_table.csv", row.names = FALSE)

In [ ]:
bmi_plot_data <- bmi_regression_result %>%
  filter(term == "measlesinfectionYes", model == "adjusted") %>%
  mutate(outcome_label = label_map[outcome])  # Add label if not already

bmi_plot <- ggplot(bmi_plot_data, aes(x = estimate, y = reorder(outcome_label, estimate), color = significant)) +
  geom_errorbarh(
    aes(xmin = estimate - std.error * 1.96, xmax = estimate + std.error * 1.96),
    height = 0,
    linewidth = 0.7
  ) +
  geom_point(size = 3) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
  scale_color_manual(values = c(`TRUE` = "darkslategray3", `FALSE` = "gray80")) +
  labs(
    title = "Effect of Measles Infection on BMI",
    x = "Beta Coefficient (95% CI)",
    y = "Outcome",
    color = "Significant"
  ) +
  my_custom_theme()

ggsave("bmi_plot.svg", plot = bmi_plot, device = svglite, width = 6, height = 4)
ggsave("bmi_plot.pdf", plot = bmi_plot, width = 7, height = 2)

In [ ]:
bmi_plot

In [ ]:
#### sex interaction